# 03 - Tabla agregada para Power BI

Agrega los 8 años limpios de beneficiarios FONASA en una sola tabla con el conteo de beneficiarios por cada combinación de dimensiones, agrega columnas de orden para renta y región y la guarda para usarla en Power BI.

**Entrada:** `base-beneficiarios-limpia/` - 8 archivos Parquet (`beneficiarios_fonasa_<año>_limpio.parquet`) generados por el notebook 02.

**Salida:** `power-bi/fonasa_beneficiarios_agregado.parquet` - un archivo Parquet con una fila por combinación de dimensiones y año, la columna `beneficiarios` y las columnas `orden_renta` y `orden_region`.

**Pasos**
1. Paso 1 - Agregar los 8 años y validar totales por año
2. Paso 2 - Agregar columnas de orden y guardar la tabla para Power BI

## Paso 1 - Agregar los 8 años y validar totales por año

Para cada año carga solo las dimensiones disponibles en `DIMENSIONES` (las columnas que faltan en 2021 se omiten), las convierte a `category` para reducir memoria y cuenta los beneficiarios por combinación de dimensiones con `groupby(...).size()`. Las 8 tablas agregadas se unen en `tabla`, lo que reduce unos 125 millones de registros individuales a una tabla manejable para Power BI.

Resultado esperado: totales por año iguales a las filas de los archivos limpios: 14.102.709 (2018), 14.841.577 (2019), 15.142.528 (2020), 15.233.814 (2021), 15.613.584 (2022), 16.229.898 (2023), 16.752.189 (2024), 17.132.611 (2025); 3.766.700 filas totales.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

carpeta_limpia = Path(r"C:\Users\ijoaq\PythonArchivos\PythonPractica\fondo-nacional-de-salud\base-beneficiarios-limpia")

DIMENSIONES = ["anio", "región_beneficiario", "comuna_beneficiario", "sexo", "edad_tramo",
               "nacionalidad", "titular_carga", "tramo_fonasa", "tipo_asegurado",
               "caracterización", "tramo_renta"]

partes = []

for archivo in sorted(carpeta_limpia.glob("*.parquet")):
    disponibles = [c for c in DIMENSIONES if c in pq.read_schema(archivo).names]
    df = pd.read_parquet(archivo, columns=disponibles)
    df = df.astype({c: "category" for c in disponibles if c != "anio"})
    agregado = df.groupby(disponibles, observed=True, dropna=False).size().reset_index(name="beneficiarios")
    print(archivo.name, "-", len(df), "personas ->", len(agregado), "filas")
    partes.append(agregado)
    del df

tabla = pd.concat(partes, ignore_index=True)

print(tabla.groupby("anio")["beneficiarios"].sum())
print("Filas totales:", len(tabla))

beneficiarios_fonasa_2018_limpio.parquet - 14102709 personas -> 489102 filas
beneficiarios_fonasa_2019_limpio.parquet - 14841577 personas -> 462201 filas
beneficiarios_fonasa_2020_limpio.parquet - 15142528 personas -> 472107 filas
beneficiarios_fonasa_2021_limpio.parquet - 15233814 personas -> 61219 filas
beneficiarios_fonasa_2022_limpio.parquet - 15613584 personas -> 568367 filas
beneficiarios_fonasa_2023_limpio.parquet - 16229898 personas -> 605218 filas
beneficiarios_fonasa_2024_limpio.parquet - 16752189 personas -> 559401 filas
beneficiarios_fonasa_2025_limpio.parquet - 17132611 personas -> 549085 filas
anio
2018    14102709
2019    14841577
2020    15142528
2021    15233814
2022    15613584
2023    16229898
2024    16752189
2025    17132611
Name: beneficiarios, dtype: int64
Filas totales: 3766700


## Paso 2 - Agregar columnas de orden y guardar la tabla para Power BI

Agrega `orden_renta` (1 a 8, de menor a mayor renta, con "Sin cotizaciones" al final) y `orden_region` (código oficial de región, 1 a 16, con "Desconocida" = 99), para que Power BI ordene estas categorías de forma lógica y no alfabética. Luego revisa que ninguna categoría haya quedado sin orden y guarda la tabla en `power-bi/fonasa_beneficiarios_agregado.parquet`.

Resultado esperado: "Regiones sin orden" y "Rentas sin orden" vacías (`[]`) y un archivo guardado de 11,2 MB.

In [2]:
orden_renta = {
    "1 a 249.999": 1, "250.000 a 499.999": 2, "500.000 a 749.999": 3, "750.000 a 999.999": 4,
    "1.000.000 a 1.249.999": 5, "1.250.000 a 1.499.999": 6, "1.500.000 o más": 7, "Sin cotizaciones": 8
}

orden_region = {
    "Región de Tarapacá": 1, "Región de Antofagasta": 2, "Región de Atacama": 3,
    "Región de Coquimbo": 4, "Región de Valparaíso": 5,
    "Región del Libertador General Bernardo O'Higgins": 6, "Región del Maule": 7,
    "Región del Biobío": 8, "Región de La Araucanía": 9, "Región de Los Lagos": 10,
    "Región Aysén del General Carlos Ibáñez del Campo": 11,
    "Región de Magallanes y de la Antártica Chilena": 12,
    "Región Metropolitana de Santiago": 13, "Región de Los Ríos": 14,
    "Región de Arica y Parinacota": 15, "Región de Ñuble": 16, "Desconocida": 99
}

tabla["orden_renta"] = tabla["tramo_renta"].map(orden_renta)
tabla["orden_region"] = tabla["región_beneficiario"].map(orden_region)

print("Regiones sin orden:", tabla.loc[tabla["orden_region"].isna(), "región_beneficiario"].unique())
print("Rentas sin orden:", tabla.loc[tabla["orden_renta"].isna() & tabla["tramo_renta"].notna(), "tramo_renta"].unique())

carpeta_powerbi = carpeta_limpia.parent / "power-bi"
carpeta_powerbi.mkdir(exist_ok=True)
ruta_salida = carpeta_powerbi / "fonasa_beneficiarios_agregado.parquet"
tabla.to_parquet(ruta_salida, index=False)

print("Guardado:", round(ruta_salida.stat().st_size / 1e6, 1), "MB")

Regiones sin orden: [], Categories (17, str): ['Desconocida', 'Región Aysén del General Carlos Ibáñez del Campo', 'Región Metropolitana de Santiago', 'Región de Antofagasta', ..., 'Región de Ñuble', 'Región del Biobío', 'Región del Libertador General Bernardo O'Higgins', 'Región del Maule']
Rentas sin orden: [], Categories (8, str): ['1 a 249.999', '1.000.000 a 1.249.999', '1.250.000 a 1.499.999', '1.500.000 o más', '250.000 a 499.999', '500.000 a 749.999', '750.000 a 999.999', 'Sin cotizaciones']
Guardado: 11.2 MB
